# Data Balancing

Notebook này xử lý class imbalance:
- Undersample classes có quá nhiều ảnh
- Augment classes có quá ít ảnh
- Target: ~2000 ảnh/class

## Setup & Load Config

In [ ]:
# Import config
import sys
sys.path.append('..')  # Thêm parent directory vào path

from config import config

# Import libraries
import os
import shutil
import random
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

# Print config
config.print_config()

# Set variables từ config
SOURCE_DIR = config.DATA_DIR
TARGET_DIR = config.BALANCED_DIR
TARGET_COUNT = config.TARGET_COUNT

print(f"\n Source: {SOURCE_DIR}")
print(f" Target: {TARGET_DIR}")
print(f" Target count: {TARGET_COUNT} images/class")

##  Data Balancing Process

In [ ]:
# Xóa thư mục cũ nếu có
if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)
os.makedirs(TARGET_DIR)

# Setup augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Get categories
categories = [d for d in os.listdir(SOURCE_DIR) if os.path.isdir(os.path.join(SOURCE_DIR, d))]

print("⚙️  Bắt đầu quá trình cân bằng dữ liệu...\n")

for cat in categories:
    src_cat_path = os.path.join(SOURCE_DIR, cat)
    dst_cat_path = os.path.join(TARGET_DIR, cat)
    os.makedirs(dst_cat_path, exist_ok=True)
    
    all_imgs = os.listdir(src_cat_path)
    current_count = len(all_imgs)
    
    if cat == 'clothes':
        # Undersampling: Lấy ngẫu nhiên TARGET_COUNT ảnh
        selected_imgs = random.sample(all_imgs, TARGET_COUNT)
        for img_name in selected_imgs:
            shutil.copy(os.path.join(src_cat_path, img_name), os.path.join(dst_cat_path, img_name))
        print(f" {cat}: Undersampling từ {current_count} xuống {TARGET_COUNT}")
    
    elif current_count >= 1500:
        # Giữ nguyên nếu đã đủ số lượng mong muốn
        for img_name in all_imgs:
            shutil.copy(os.path.join(src_cat_path, img_name), os.path.join(dst_cat_path, img_name))
        print(f" {cat}: Giữ nguyên {current_count} ảnh")
    
    else:
        # Oversampling + Augmentation
        # Copy ảnh gốc trước
        for img_name in all_imgs:
            shutil.copy(os.path.join(src_cat_path, img_name), os.path.join(dst_cat_path, img_name))
        
        # Augment thêm để đạt TARGET_COUNT
        needed = TARGET_COUNT - current_count
        aug_per_img = (needed // current_count) + 1
        count = 0
        
        for img_name in all_imgs:
            if count >= needed:
                break
            
            img = load_img(os.path.join(src_cat_path, img_name))
            x = img_to_array(img)
            x = x.reshape((1,) + x.shape)
            
            i = 0
            for batch in datagen.flow(x, batch_size=1, save_to_dir=dst_cat_path, 
                                       save_prefix='aug', save_format='jpeg'):
                i += 1
                count += 1
                if i >= aug_per_img or count >= needed:
                    break
        
        final_count = len(os.listdir(dst_cat_path))
        print(f" {cat}: Augment từ {current_count} lên {final_count}")

print("\n Hoàn tất! Dữ liệu cân bằng đã sẵn sàng tại:", TARGET_DIR)

## Kiểm tra kết quả

In [ ]:
# Kiểm tra lại số lượng sau khi cân bằng
final_stats = {cat: len(os.listdir(os.path.join(TARGET_DIR, cat))) for cat in categories}

plt.figure(figsize=(12, 5))
sns.barplot(x=list(final_stats.keys()), y=list(final_stats.values()))
plt.title('Số lượng ảnh sau khi cân bằng (Balanced Dataset)')
plt.xlabel('Category')
plt.ylabel('Number of Images')
plt.axhline(y=TARGET_COUNT, color='r', linestyle='--', label=f'Target: {TARGET_COUNT}')
plt.legend()
plt.xticks(rotation=45)
plt.show()

# In thống kê
print("\n Thống kê sau khi cân bằng:")
for cat, count in sorted(final_stats.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cat:15s}: {count:,} images")

print(f"\n Tổng số ảnh: {sum(final_stats.values()):,}")
print(f" Trung bình: {sum(final_stats.values()) // len(final_stats):,} images/class")

##  Kết luận

-  Đã cân bằng dữ liệu
-  Mỗi class có ~2000 ảnh